# 07. 전류 이상치 탐지 (I1, I2, I3)

## 이상치 기준
- 물리적 기준: 전류 < 0A (단방향 소비 계량기에서 음수 전류는 불가)
- 통계적 기준: 계량기별 일별 평균값 기준 평균 ± 3σ 초과
- 발전/양방향 계량기는 물리적 기준 제외 (부호 규약상 음수 정상)

In [1]:
import sys
from pathlib import Path

ROOT = Path('/home/aceya/EMS')
sys.path.insert(0, str(ROOT / 'src'))

import pandas as pd
import numpy as np
from ems.db import load_env, connect

load_env()

START = '2018-01-01'
END   = '2024-01-01'

BIDIRECTIONAL = {
    'H1.Z20', 'H1.ZE20',
    'V.Z84', 'V.ZE84',
    'V.Z81', 'V.Z82',
    'H1.Z310', 'H2.Z311', 'H3.Z312',
    'H1.Z29', 'H1.Z28',
}

ALL_METERS = [
    'H1.Z10', 'H1.Z11', 'H1.Z12', 'H1.Z13', 'H1.Z14', 'H1.Z15', 'H1.Z16',
    'H1.Z17', 'H1.Z18', 'H1.Z19', 'H1.Z20', 'H1.Z21', 'H1.Z22', 'H1.Z23',
    'H1.Z24', 'H1.Z25', 'H1.Z26', 'H1.Z27', 'H1.Z28', 'H1.Z29', 'H1.Z310',
    'H1.ZE20', 'H2.T.Z30', 'H2.T.Z31', 'H2.T.Z32', 'H2.T.Z33', 'H2.T.Z34',
    'H2.Z311', 'H2.Z35', 'H2.Z64', 'H2.Z65', 'H2.Z66', 'H2.Z67', 'H2.Z68',
    'H2.Z69', 'H2.Z70', 'H2.ZE64', 'H2.ZE65', 'H2.ZE66', 'H2.ZE67', 'H2.ZE74',
    'H3.Z312', 'H3.Z40', 'H3.Z41', 'H3.Z42', 'H4.Z50', 'H4.Z51', 'H4.ZE50',
    'H4.ZE51', 'V.Z81', 'V.Z82', 'V.Z84', 'V.ZE84'
]

save_dir = ROOT / 'outputs/tables/anomaly'
save_dir.mkdir(parents=True, exist_ok=True)
print('설정 완료')

설정 완료


In [2]:
def fetch_daily(meter_urn, measurement):
    sql = """
        SELECT
            DATE(ts AT TIME ZONE 'Europe/Berlin') AS day,
            MIN(value) AS min_val,
            MAX(value) AS max_val,
            AVG(value) AS avg_val,
            COUNT(*) AS cnt
        FROM ems.cr_measurement_1h
        WHERE meter_urn = %s
          AND measurement = %s
          AND ts >= %s
          AND ts <  %s
        GROUP BY 1
        ORDER BY 1
    """
    with connect() as conn:
        df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))
    df['day'] = pd.to_datetime(df['day'])
    return df


def detect_anomaly(meter, measurement):
    df = fetch_daily(meter, measurement)
    if df.empty:
        return None

    results = []

    if meter not in BIDIRECTIONAL:
        physical = df[df['min_val'] < 0].copy()
        physical['anomaly_type'] = '물리적이상(음수전류)'
        physical['criterion'] = 'min_val < 0A'
        if len(physical) > 0:
            results.append(physical)

    mean_val = df['avg_val'].mean()
    std_val  = df['avg_val'].std()
    upper = mean_val + 3 * std_val
    lower = mean_val - 3 * std_val
    stat = df[(df['avg_val'] > upper) | (df['avg_val'] < lower)].copy()
    stat['anomaly_type'] = '통계적이상(3sigma)'
    stat['criterion'] = f'mean={mean_val:.2f}, sigma={std_val:.2f}, lower={lower:.2f}, upper={upper:.2f}'
    if len(stat) > 0:
        results.append(stat)

    if not results:
        return None

    result = pd.concat(results).drop_duplicates('day').sort_values('day')
    result['meter'] = meter
    result['measurement'] = measurement
    return result[['meter', 'measurement', 'day', 'min_val', 'max_val', 'avg_val', 'anomaly_type', 'criterion']]

In [3]:
all_results = []

for meter in ALL_METERS:
    for meas in ['I1', 'I2', 'I3']:
        result = detect_anomaly(meter, meas)
        if result is not None and len(result) > 0:
            print(f'{meter} {meas}: {len(result)}건')
            all_results.append(result)

if all_results:
    final = pd.concat(all_results, ignore_index=True)
    final.to_csv(save_dir / 'anomaly_current.csv', index=False)
    print(f'\n총 {len(final)}건 저장 완료')
else:
    print('이상치 없음')

/tmp/ipykernel_100795/3433663356.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


H1.Z10 I1: 23건


/tmp/ipykernel_100795/3433663356.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


H1.Z10 I2: 68건


/tmp/ipykernel_100795/3433663356.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


...

V.Z82 I2: 78건


/tmp/ipykernel_100795/3433663356.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))


V.Z82 I3: 46건


/tmp/ipykernel_100795/3433663356.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))
/tmp/ipykernel_100795/3433663356.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))
/tmp/ipykernel_100795/3433663356.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=(meter_urn, measurement, START, END))
/tmp/ipykernel_100795/3433663356.py:18: UserWarning:

V.ZE84 I3: 1건

총 9569건 저장 완료


In [4]:
if all_results:
    summary = final.groupby(['meter', 'measurement', 'anomaly_type']).agg(
        건수=('day', 'count'),
        시작일=('day', 'min'),
        종료일=('day', 'max'),
        min_val=('min_val', 'min'),
        max_val=('max_val', 'max'),
        criterion=('criterion', 'first')
    ).reset_index()
    summary.to_csv(save_dir / 'anomaly_current_summary.csv', index=False)
    print(summary.to_string())

        meter measurement   anomaly_type    건수        시작일        종료일       min_val     max_val                                               criterion
0      H1.Z10          I1  통계적이상(3sigma)    23 2018-08-29 2022-07-07  4.532667e+00   45.439833         mean=6.02, sigma=2.45, lower=-1.35, upper=13.38
1      H1.Z10          I2  통계적이상(3sigma)    68 2018-01-01 2023-06-25  2.745869e-01   43.574833          mean=8.76, sigma=2.47, lower=1.37, upper=16.16
2      H1.Z10          I3  통계적이상(3sigma)    20 2018-08-29 2022-02-09  4.344167e+00   43.309333         mean=6.63, sigma=2.46, lower=-0.75, upper=14.01
3      H1.Z11          I1  통계적이상(3sigma)    36 2018-05-28 2023-08-23  4.585669e-01  139.641833      mean=11.45, sigma=14.49, lower=-32.02, upper=54.92
4      H1.Z11          I2  통계적이상(3sigma)    34 2018-05-28 2023-08-18  4.840347e-01  141.893417      mean=11.66, sigma=14.75, lower=-32.58, upper=55.91
5      H1.Z11          I3  통계적이상(3sigma)    36 2018-05-28 2023-08-23  0.000000e+00  139.663333